# PolyWin R2 — v22: BERT-arm (BPE + SMILES MLM) on the P14 blend

## What this kernel does
* **Level-0 (P14, frozen, bit-identical):** verbatim P14 sources from
  `mt_gnn_v2.py` (CORE_A graph feats + GINE encoder pretrained on the entire
  PI1M archive, then CORE_B: leak-safe twins + MT-GNN fold OOF + GBM trio
  stack). Seeds 42/999/2025, 5 folds, all params unchanged from P14.
* **v22 arm:** a fully in-notebook, label-free BERT-style SMILES encoder:
  BPE tokenizer (trained on the PI1M sample) -> masked-token MLM pretraining
  (`BertEncoder`, pure torch) -> mean-pooled embeddings -> per-target fold-safe
  Ridge heads (`compute_bert_arm`, GroupKFold on canonical smiles) producing
  `oof_bert` / `test_bert`.
* **Blend:** P14 fold-safe per-target alpha sweep on the 3 arms (gbm, mt, bert)
  via `blend_narm_oof`; the P14 2-arm reference is recomputed in-cell with
  `_p14_2arm_oof` so gate 2 compares like-for-like.
* **Gates (pre-registered, do NOT soften):** gates 0-3 mirror v21:
  * gate 1 (leak audit): BERT-arm features vs true other-target labels exact
    match count must be 0
  * gate 2 (OOF gain): mean over {eps,nc,ei} AND overall >= P14 reference
    + +0.0015 (soft) / +0.003 (strong)
  * gate 3 (worst-target): every per-target delta >= -0.003
  * Verdict: `GATE: PASS -> v22 proceeds` or `GATE: FAIL -> P14 stays final`.
* Submission: `submission_v22.csv` (`id,target`, P14 format) written ONLY on
  PASS; `v22_blend_report.csv` is always written.

Only OSI-approved libs: PyTorch, RDKit, scikit-learn, LightGBM, CatBoost, XGBoost.


In [ ]:
import os, sys, time, gc, random, warnings
import subprocess, importlib.util

def ensure_pkg(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print("installing", pkg, flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--disable-pip-version-check", pkg])

for _p, _n in [("rdkit", "rdkit"), ("torch_geometric", "torch_geometric"),
               ("lightgbm", "lightgbm"), ("catboost", "catboost"),
               ("xgboost", "xgboost"), ("scipy", "scipy")]:
    ensure_pkg(_p, _n)

# --- CUDA probe / repair identical to v14/v16 (P100 sm_60 needs torch 2.5.1) ---
_probe = ('import torch;' + 'a=torch.zeros(4,device="cuda");b=a+1;torch.cuda.synchronize();print("OK")')
def _force_cuda():
    try:
        _r = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                            text=True, timeout=600)
    except Exception:
        _r = None
    if _r is not None and _r.returncode == 0 and "OK" in (_r.stdout or ""):
        return
    print("CUDA kernel missing; installing torch 2.5.1 (cu121, supports P100 sm_60)...", flush=True)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--no-cache-dir", "--index-url",
                               "https://download.pytorch.org/whl/cu121",
                               "torch==2.5.1"], timeout=1800)
        _r2 = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                             text=True, timeout=600)
        print("post-reinstall probe rc:", _r2.returncode, flush=True)
    except Exception as _e:
        print("torch reinstall errored:", repr(_e)[:200], flush=True)
if os.path.exists("/kaggle"):
    _force_cuda()

# --- CUDA probe (no reinstall needed; Kaggle GPU has a working build) ---
def _cuda_ok():
    try:
        if not torch.cuda.is_available():
            return False
        a = torch.zeros(4, device="cuda"); b = a + 1; torch.cuda.synchronize(); del a, b
        return True
    except Exception:
        return False

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, rdMolDescriptors, Crippen, GraphDescriptors
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if _cuda_ok() else "cpu"
print("device:", DEVICE, flush=True)
GLOBAL_FOLDS = 5
MAX_EPOCHS = 120
PATIENCE = 20
EARLY_HOLDOUT = 0.15
BS = 256
LR = 1e-3
if DEVICE == "cpu":
    GLOBAL_FOLDS = min(GLOBAL_FOLDS, 2)
    MAX_EPOCHS = min(MAX_EPOCHS, 12)
    BS = 256
    PATIENCE = min(PATIENCE, 6)
PRETRAIN_EPOCHS = 10
PRETRAIN_SAMPLE = 2000000
GNN_SEEDS = "42,999,2025"
os.environ["GNN_SEEDS"] = GNN_SEEDS

# v22 BERT-arm config ------------------------------------------------
V22_PI_COUNT = -1
V22_D = 384
V22_LAYERS = 6
V22_HEADS = 8
V22_EPOCHS = 1
V22_BPE_SUBSET = 150000
V22_VOCAB = 4000
V22_SEED = 42

# Kaggle-only data source: the competition input tree. Base is the /kaggle/input
# root so find_input probes every mount layout (direct, <slug>, competitions/<slug>).
if os.path.exists("/kaggle"):
    INP = "/kaggle/input"
    WORK = "/kaggle/working"
    os.environ.setdefault("SMOKE", "1" if SMOKE else "0")
else:
    INP = "official_dataset"
    WORK = os.path.join("vault", "pipeline_out_v22")
os.makedirs(WORK, exist_ok=True)
PRETRAINED = os.path.join(WORK, "pretrained_encoder.pt")
OUT = WORK

print("----- v22 CONFIG -----", flush=True)
print("folds:", GLOBAL_FOLDS, "| GNN_SEEDS:", GNN_SEEDS,
      "| PRETRAIN_SAMPLE:", PRETRAIN_SAMPLE, flush=True)
print("v22: PI_COUNT", V22_PI_COUNT, "d", V22_D, "layers", V22_LAYERS,
      "heads", V22_HEADS, "epochs", V22_EPOCHS, "bpe_subset", V22_BPE_SUBSET,
      "vocab", V22_VOCAB, "seed", V22_SEED, flush=True)
print("device:", DEVICE, "| SMOKE:", SMOKE, "| out:", OUT, flush=True)


In [ ]:
from sklearn.linear_model import Ridge

TARGETS = ["eea", "egb", "egc", "ei", "eps", "nc", "tg"]
TARGET_IDX = {t: i for i, t in enumerate(TARGETS)}


## 1. Data — current-round CSVs, canonicalize, compute descriptors + fingerprints

In [ ]:
def find_input(base, name):
    for p in [os.path.join(base, name), os.path.join(base, "ppp-round-2", name),
              os.path.join(base, "competitions", "ppp-round-2", name)]:
        if os.path.exists(p):
            return p
    return None

def canonical(s):
    if not isinstance(s, str):
        return None, None, None
    m = Chem.MolFromSmiles(s)
    if m is None:
        return None, None, None
    try:
        c = Chem.MolToSmiles(m)
        ik = Chem.MolToInchiKey(m)
    except Exception:
        return Chem.MolToSmiles(m), None, None
    return c, ik, m

def canon_fast(s):
    """MolToSmiles only — identical string to canonical(s)[0] for the PI1M
    pretrain corpus, avoids ~N MolToInchiKey computations, no result change."""
    if not isinstance(s, str):
        return None
    try:
        m = Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(m) if m is not None else None
    except Exception:
        return None

def feats(m):
    if m is None:
        return [np.nan] * 35
    try:
        Chem.rdPartialCharges.ComputeGasteigerCharges(m)
        gasteiger = [a.GetDoubleProp('_GasteigerCharge') for a in m.GetAtoms()]
        g_mean = np.mean(gasteiger)
        g_std = np.std(gasteiger) if len(gasteiger) > 1 else 0.0
        g_min = np.min(gasteiger); g_max = np.max(gasteiger)
    except Exception:
        g_mean = g_std = g_min = g_max = 0.0
    atoms = m.GetAtoms()
    n_total = len(atoms) if atoms else 1
    elem_counts = {}
    for a in atoms:
        sym = a.GetSymbol()
        elem_counts[sym] = elem_counts.get(sym, 0) + 1
    frac_C = elem_counts.get("C", 0) / n_total
    frac_N = elem_counts.get("N", 0) / n_total
    frac_O = elem_counts.get("O", 0) / n_total
    frac_S = elem_counts.get("S", 0) / n_total
    frac_F = elem_counts.get("F", 0) / n_total
    n_hetero = sum(v for k, v in elem_counts.items() if k not in ("C", "H"))
    frac_hetero = n_hetero / n_total
    bonds = m.GetBonds()
    n_bonds = len(bonds) if bonds else 1
    bond_counts = {"SINGLE": 0, "DOUBLE": 0, "TRIPLE": 0, "AROMATIC": 0}
    for b in bonds:
        bt = b.GetBondType().name
        if bt in bond_counts:
            bond_counts[bt] += 1
    ratio_single = bond_counts["SINGLE"] / n_bonds
    ratio_double = bond_counts["DOUBLE"] / n_bonds
    ratio_aromatic = bond_counts["AROMATIC"] / n_bonds
    return [
        Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
        Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m),
        Descriptors.RingCount(m), Descriptors.NumAromaticRings(m),
        Descriptors.NumAliphaticRings(m), Descriptors.NumSaturatedRings(m),
        Descriptors.NumRotatableBonds(m), rdMolDescriptors.CalcNumHeavyAtoms(m),
        Descriptors.NumHeteroatoms(m), Descriptors.FractionCSP3(m),
        Crippen.MolMR(m), rdMolDescriptors.CalcNumBridgeheadAtoms(m),
        rdMolDescriptors.CalcNumSpiroAtoms(m),
        rdMolDescriptors.CalcNumAromaticAtoms(m) if hasattr(rdMolDescriptors, "CalcNumAromaticAtoms") else Descriptors.NumAromaticRings(m),
        GraphDescriptors.BalabanJ(m), GraphDescriptors.Ipc(m),
        rdMolDescriptors.CalcNumLipinskiHBA(m), rdMolDescriptors.CalcNumLipinskiHBD(m),
        rdMolDescriptors.CalcNumAtomStereoCenters(m),
        g_mean, g_std, g_min, g_max,
        frac_C, frac_N, frac_O, frac_S, frac_F, frac_hetero,
        ratio_single, ratio_double, ratio_aromatic,
    ]

FNAMES = ["MolWt", "LogP", "TPSA", "HDon", "HAccep", "RingCnt", "AroRing", "AliRing", "SatRing",
          "RotB", "HeavyAt", "HeteroAt", "FracCSP3", "MR", "Bridge", "Spiro", "AroAt",
          "BalabanJ", "Ipc", "LipHBA", "LIHBD", "Stereo",
          "GMean", "GStd", "GMin", "GMax",
          "FracC", "FracN", "FracO", "FracS", "FracF", "FracHetero",
          "RatioSingle", "RatioDouble", "RatioAro"]
assert len(FNAMES) == 35

train_path = find_input(INP, "train.csv")
test_path = find_input(INP, "test.csv")
assert train_path and test_path, "train.csv / test.csv not found in " + INP

tr = pd.read_csv(train_path)
te = pd.read_csv(test_path)
print("train:", tr.shape, "test:", te.shape, flush=True)

tcpl = tr["smiles"].map(canonical)
tr["canon"], tr["inchikey"], _ = zip(*tcpl)
tepl = te["smiles"].map(canonical)
te["canon"], te["inchikey"], _ = zip(*tepl)

tr_f = np.array(tr["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
te_f = np.array(te["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
tr[FNAMES] = tr_f
te[FNAMES] = te_f
print("descriptors done", flush=True)

trf = tr.dropna(subset=["target"]).copy()
tef = te.copy()

FEAT_COLS = [c for c in trf.columns if c not in
            ("smiles", "target", "target_type", "canon", "inchikey", "id")]
print("FEAT_COLS:", len(FEAT_COLS), flush=True)


In [ ]:
def add_fingerprints(df):
    morgan = np.zeros((len(df), 2048), dtype=np.float32)
    maccs = np.zeros((len(df), 167), dtype=np.float32)
    ap = np.zeros((len(df), 1024), dtype=np.float32)
    tt = np.zeros((len(df), 1024), dtype=np.float32)
    ap_gen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=1024)
    tt_gen = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=1024)
    for i, s in enumerate(df["smiles"]):
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        morgan[i] = np.frombuffer(AllChem.GetMorganFingerprintAsBitVect(
            m, 2, nBits=2048).ToBitString().encode(), "u1") - ord("0")
        maccs[i] = np.frombuffer(MACCSkeys.GenMACCSKeys(m).ToBitString().encode(),
                                 "u1") - ord("0")
        ap[i] = np.frombuffer(ap_gen.GetFingerprint(m).ToBitString().encode(),
                              "u1") - ord("0")
        tt[i] = np.frombuffer(tt_gen.GetFingerprint(m).ToBitString().encode(),
                              "u1") - ord("0")
    return morgan, maccs, ap, tt

F32_MAX = np.finfo(np.float32).max

def clean_feats(df):
    D = np.clip(df[FEAT_COLS].values, -F32_MAX, F32_MAX)
    for j in range(D.shape[1]):
        col = D[:, j]
        med = np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0.0
        col[~np.isfinite(col)] = med
    return D.astype(np.float32)

D_tr = clean_feats(trf)
D_te = clean_feats(tef)
mor_tr, mc_tr, ap_tr, tt_tr = add_fingerprints(trf)
mor_te, mc_te, ap_te, tt_te = add_fingerprints(tef)

Y = trf["target"].values.astype(np.float32)
T = trf["target_type"].values
G = trf["canon"].values.astype(str)
idx_of_target = {t: np.where(T == t)[0] for t in TARGETS}

X = np.hstack([D_tr, mor_tr, mc_tr, ap_tr, tt_tr]).astype(np.float32)
Xs = StandardScaler().fit(X).transform(X).astype(np.float32)

Xte = np.hstack([D_te, mor_te, mc_te, ap_te, tt_te]).astype(np.float32)
Xtes = StandardScaler().fit(X).transform(Xte).astype(np.float32)

print("train:", X.shape, "test:", Xte.shape, "targets:", TARGETS, flush=True)


## 2. Level-0 sources (verbatim from mt_gnn_v2.py: graph feats + GINE + MT-GNN)

In [ ]:
# Graph featurization (MUST match the v10 pretrain kernel so the saved
# pretrained_encoder.pt loads into the same GINEEncoder).
# =====================================================================
ATOM_SYMBOLS = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "Si", "P", "OTHER"]
HYBRIDIZATIONS = ["SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"]
BOND_TYPES = ["SINGLE", "DOUBLE", "TRIPLE", "AROMATIC"]


def one_hot(value, choices):
    vec = [0.0] * len(choices)
    idx = choices.index(value) if value in choices else len(choices) - 1
    vec[idx] = 1.0
    return vec


def atom_features(atom):
    return (one_hot(atom.GetSymbol(), ATOM_SYMBOLS)
            + one_hot(atom.GetHybridization().name, HYBRIDIZATIONS)
            + [atom.GetIsAromatic() * 1.0, atom.IsInRing() * 1.0,
               atom.GetDegree() / 4.0, atom.GetTotalNumHs() / 4.0,
               atom.GetFormalCharge() / 2.0])


N_ATOM_FEATS = len(ATOM_SYMBOLS) + len(HYBRIDIZATIONS) + 5
N_BOND_FEATS = len(BOND_TYPES) + 2


def bond_features(bond):
    return one_hot(bond.GetBondType().name, BOND_TYPES) + [
        bond.GetIsConjugated() * 1.0, bond.IsInRing() * 1.0]


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() < 2:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]
    if len(edge_index) == 0:
        edge_index = [[0, 0]]; edge_attr = [[0.0] * N_BOND_FEATS]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


def build_graphs(df, has_target=True):
    out = {}
    freq = df["target_type"].value_counts(normalize=True)
    for row_id, row in zip(df.index, df.itertuples()):
        g = smiles_to_graph(row.smiles)
        if g is None:
            continue
        g.row_id = row_id
        g.smiles = row.smiles
        if has_target:
            g.target_idx = torch.tensor([TARGET_IDX[row.target_type]], dtype=torch.long)
            g.y = torch.tensor([float(row.target)], dtype=torch.float)
            g.w = torch.tensor([1.0 / freq[row.target_type]], dtype=torch.float)
        out[row_id] = g
    return out


def to_pyg(graphs):
    if isinstance(graphs, dict):
        graphs = list(graphs.values())
    return Batch.from_data_list(graphs)


t0 = time.time()
train_graphs = build_graphs(trf, has_target=True)
test_graphs = build_graphs(tef, has_target=False)
print(f"graphs: {len(train_graphs)} train, {len(test_graphs)} test "
      f"({time.time()-t0:.0f}s)", flush=True)


# =====================================================================
# Shared encoder + multi-task trunk (same GINEEncoder as v10 kernel).
# =====================================================================
class GINEEncoder(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4, dropout=0.2):
        super().__init__()
        self.atom_encoder = nn.Linear(n_atom_feats, hidden)
        self.bond_encoder = nn.ModuleList(
            [nn.Linear(n_bond_feats, hidden) for _ in range(n_layers)])
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        h = self.atom_encoder(x)
        for conv, bn, bond_enc in zip(self.convs, self.bns, self.bond_encoder):
            e = bond_enc(edge_attr)
            h = conv(h, edge_index, e)
            h = bn(h); h = F.relu(h); h = F.dropout(h, p=self.dropout,
                                                    training=self.training)
        return h


class MTGNN(nn.Module):
    """Shared trunk + per-target heads. Optional cross-target twin features
    are concatenated to the pooled embedding before the shared trunk."""

    def __init__(self, n_atom_feats, n_bond_feats, n_twin=0, hidden=128,
                 n_layers=4, dropout=0.2):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden,
                                   n_layers, dropout)
        pool_in = hidden * 2 + n_twin
        self.trunk = nn.Sequential(
            nn.Linear(pool_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(dropout))
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
                          nn.Linear(64, 1))
            for _ in TARGETS])

    def forward(self, data, twin=None):
        h = self.encoder(data.x, data.edge_index, data.edge_attr)
        pooled = torch.cat([global_mean_pool(h, data.batch),
                            global_add_pool(h, data.batch)], dim=1)
        if twin is not None and twin.size(1) > 0:
            pooled = torch.cat([pooled, twin.to(pooled.device)], dim=1)
        ht = self.trunk(pooled)
        out = torch.empty(data.batch.max() + 1, len(TARGETS), device=h.device)
        for i, head in enumerate(self.heads):
            out[:, i] = head(ht)[:, 0]
        return out

    def load_encoder(self, state_dict):
        enc = {k[len("encoder."):]: v for k, v in state_dict.items()
               if k.startswith("encoder.")}
        missing, unexpected = self.encoder.load_state_dict(enc, strict=False)
        print(f"  encoder init: missing={len(missing)} unexpected={len(unexpected)}",
              flush=True)


# =====================================================================

## 3. Pretrain the GINE encoder on PI1M

In [ ]:
pl_path = find_input(INP, "PI1M.csv")
pl = []
if pl_path:
    pldf = pd.read_csv(pl_path)
    smi_col = "SMILES" if "SMILES" in pldf.columns else "smiles"
    pldf = pldf[[smi_col]].rename(columns={smi_col: "smiles"})
    t0pl = time.time()
    pldf["canon"] = pldf["smiles"].map(canon_fast)
    print(f"PI1M canonicalized in {time.time()-t0pl:.0f}s "
          f"({len(pldf)} rows, parsed {pldf['canon'].notna().sum()})", flush=True)
    pldf = pldf.dropna(subset=["canon"])
    pl = pldf.drop_duplicates("canon")["smiles"].tolist()
    print("PI1M unique canons:", len(pl), flush=True)
    rng = np.random.RandomState(SEED); rng.shuffle(pl)
    pl = pl[:PRETRAIN_SAMPLE]
    print("PI1M full-PI1M pretraining corpus:", len(pl), "SMILES", flush=True)
else:
    print("no PI1M: pretraining skipped", flush=True)

def build_pretrain_graphs_chunked(smiles_list, chunk=50000):
    graphs = []
    t0 = time.time()
    for c0 in range(0, len(smiles_list), chunk):
        chunk_g = []
        for smi in smiles_list[c0:c0+chunk]:
            g = smiles_to_graph(smi)
            if g is not None:
                chunk_g.append(g)
        graphs.extend(chunk_g)
        del chunk_g
        gc.collect()
        print(f"  graphs {len(graphs)}/{len(smiles_list)} "
              f"({time.time()-t0:.0f}s)", flush=True)
    return graphs

pl_graphs = build_pretrain_graphs_chunked(pl) if pl else []
print("pretraining graphs (full PI1M):", len(pl_graphs), flush=True)

from torch_geometric.loader import DataLoader

class PretrainedEncoder(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4,
                 mask_atom=0.15, mask_bond=0.20):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden, n_layers)
        self.atom_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_atom_feats))
        self.bond_proj = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_bond_feats))
        self.mask_atom = mask_atom; self.mask_bond = mask_bond

    def forward(self, x, edge_index, edge_attr, batch):
        n = x.size(0); m = edge_index.size(1)
        atom_mask = torch.rand(n, device=x.device) < self.mask_atom
        bond_mask = torch.rand(m, device=x.device) < self.mask_bond
        x_c = x.clone(); x_c[atom_mask] = 0.0
        ea_c = edge_attr.clone(); ea_c[bond_mask] = 0.0
        h = self.encoder(x_c, edge_index, ea_c)
        if atom_mask.any():
            atom_loss = F.mse_loss(self.atom_proj(h[atom_mask]), x[atom_mask])
        else:
            atom_loss = torch.zeros((), device=x.device)
        if bond_mask.any():
            src = h[edge_index[0, bond_mask]]; dst = h[edge_index[1, bond_mask]]
            if src.numel() > 0:
                bond_loss = F.mse_loss(self.bond_proj(torch.cat([src, dst], dim=1)), edge_attr[bond_mask])
            else:
                bond_loss = torch.zeros((), device=x.device)
        else:
            bond_loss = torch.zeros((), device=x.device)
        return atom_loss, bond_loss

def pretrain(epochs=PRETRAIN_EPOCHS, batch_size=1024, lr=1e-3):
    if len(pl_graphs) == 0:
        print("No PI1M graphs - pretraining skipped", flush=True)
        return None
    model = PretrainedEncoder(N_ATOM_FEATS, N_BOND_FEATS).to(DEVICE)
    loader = DataLoader(pl_graphs, batch_size=batch_size, shuffle=True,
                        pin_memory=(DEVICE == "cuda"))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best = np.inf; best_state = None; t0 = time.time()
    for epoch in range(epochs):
        model.train(); tot_a = 0.0; tot_b = 0.0; nbl = 0
        for batch in loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            a_loss, b_loss = model(batch.x, batch.edge_index, batch.edge_attr, batch)
            loss = a_loss + 0.5 * b_loss
            loss.backward(); opt.step()
            tot_a += a_loss.item(); tot_b += b_loss.item(); nbl += 1
            del batch
        va = (tot_a + 0.5 * tot_b) / max(nbl, 1)
        if va < best:
            best = va; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"pretrain ep {epoch+1}/{epochs}: loss={va:.4f} ({time.time()-t0:.0f}s)", flush=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if best_state:
        torch.save(best_state, PRETRAINED)
        print("saved pretrained_encoder.pt", flush=True)
    return best_state

print("=== P14: Pretraining GNN on PI1M ===", flush=True)
pretrained_state = pretrain()


## 4. Level-0 predictions (verbatim: leak-safe twins + MT-GNN fold OOF + GBM trio stack)

In [ ]:
# Twin source: per-target LGBM OOF (leak-safe) + fold-bagged test preds.
# twin_u(row i) = target-u LGBM's prediction on row i's features.
# =====================================================================
print("\n=== Twin source: per-target LGBM OOF (leak-safe) ===", flush=True)
lgb_test_te = np.zeros((len(Xte), len(TARGETS)), dtype=np.float32)
TARGET_MEAN = {t: float(Y[idx_of_target[t]].mean()) for t in TARGETS}


# For train, twin_u(row i) uses lgb_oof_all from the target-u LGBM. But
# lgb_oof_all is stored by row index only for target-u rows. For a row of
# target t, its target-u twin value is the OOF prediction of the model_u on
# THAT row's features - we approximate with the per-target model_u evaluated
# on every train row (OOF where available, fold-safe holdout elsewhere).
# Simplest leak-safe approach: evaluate each target-u LGBM on ALL train rows
# via a dedicated OOF-style pass below.
print("\n=== Building leak-safe twin feature matrices ===", flush=True)
twin_train = np.zeros((len(X), (len(TARGETS) - 1) * 2), dtype=np.float32)
twin_test = np.zeros((len(Xte), (len(TARGETS) - 1) * 2), dtype=np.float32)
col_map = {}
for u in TARGETS:
    col = 0
    for t2 in TARGETS:
        if t2 == u:
            continue
        col_map[(u, t2)] = (col, col + 1)
        col += 2
def leak_safe_oof_scores():
    """For each target u, score every train row with a model trained on a
    canon-group that excludes that row (grouped OOF across all targets)."""
    scores = np.full((len(X), len(TARGETS)), np.nan, dtype=np.float32)
    # canon -> group id per row, using one global fold assignment
    gkf = GroupKFold(n_splits=GLOBAL_FOLDS)
    row_fold = np.zeros(len(X), dtype=int)
    for f, (_, va) in enumerate(gkf.split(Xs, Y, G)):
        row_fold[va] = f
    for u in TARGETS:
        for f in range(GLOBAL_FOLDS):
            in_fold = np.where(row_fold == f)[0]
            out_fold = np.setdiff1d(np.arange(len(X)), in_fold)
            idx_u_out = np.intersect1d(out_fold, idx_of_target[u])
            if len(idx_u_out) == 0:
                continue
            fit_ids, ho_ids = train_test_split(idx_u_out,
                                               test_size=EARLY_HOLDOUT,
                                               random_state=SEED)
            m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05,
                                  num_leaves=15, min_child_samples=10,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=SEED, verbose=-1)
            m.fit(Xs[fit_ids], Y[fit_ids], eval_set=[(Xs[ho_ids], Y[ho_ids])])
            scores[in_fold, TARGET_IDX[u]] = m.predict(Xs[in_fold])
            # test bag
            lgb_test_te[:, TARGET_IDX[u]] += m.predict(Xtes) / GLOBAL_FOLDS
    return scores, lgb_test_te


twin_scores, lgb_test_te = leak_safe_oof_scores()
for t in TARGETS:
    for u in TARGETS:
        if u == t:
            continue
        iu = TARGET_IDX[u]
        c0, c1 = col_map[(t, u)]
        impute = TARGET_MEAN[u]
        v = twin_scores[:, iu]
        miss = np.isnan(v).astype(np.float32)
        v = np.where(miss, impute, v)
        twin_train[:, c0] = v; twin_train[:, c1] = miss
        # test: fold-bagged model_u prediction, always available
        tv = lgb_test_te[:, iu]
        tmiss = np.isnan(tv).astype(np.float32)
        tv = np.where(tmiss, impute, tv)
        twin_test[:, c0] = tv; twin_test[:, c1] = tmiss
print("twin matrices:", twin_train.shape, twin_test.shape, flush=True)


# =====================================================================
# MT-GNN fold-safe OOF + test bag
# =====================================================================
def early_split(fit_ids):
    uniq_g = np.unique(G[fit_ids])
    uniq_f, uniq_h = train_test_split(uniq_g, test_size=EARLY_HOLDOUT,
                                      random_state=SEED)
    return (fit_ids[np.isin(G[fit_ids], uniq_f)],
            fit_ids[np.isin(G[fit_ids], uniq_h)])


row_to_graph = {g.row_id: g for g in train_graphs.values()}
print("\n=== MT-GNN v2 (pretrained-init trunk + twins) ===", flush=True)
pretrained_state = torch.load(PRETRAINED, map_location="cpu") if os.path.exists(
    PRETRAINED) else None
if pretrained_state is not None:
    print("loaded pretrained_encoder.pt", flush=True)

GNN_SEEDS = [int(s) for s in os.environ.get("GNN_SEEDS", "42").split(",") if s.strip()]


def run_gnn_seed(seed):
    """One seed's MT-GNN: fold-safe GroupKFold OOF + fold-bagged test preds.
    Returns (mt_oof_all, mt_test) in raw scale. Identical math to the v13 run
    except torch/np/random seeding are reset per seed."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    n_twin = twin_train.shape[1]
    mt_oof_all = np.full(len(X), np.nan, dtype=np.float32)
    mt_test_folds = np.zeros((len(Xte), GLOBAL_FOLDS), dtype=np.float32)
    for f, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=GLOBAL_FOLDS).split(
            Xs, Y, G)):
        t0f = time.time()
        stats = {}
        y_norm = np.empty(len(tr_idx), dtype=np.float32)
        for t in TARGETS:
            mask = (T[tr_idx] == t)
            if mask.sum() > 0:
                mu, sd = Y[tr_idx][mask].mean(), Y[tr_idx][mask].std() + 1e-6
                stats[t] = (mu, sd)
                y_norm[mask] = (Y[tr_idx][mask] - mu) / sd
        fit_ids, ho_ids = early_split(tr_idx)
        model = MTGNN(N_ATOM_FEATS, N_BOND_FEATS, n_twin=n_twin).to(DEVICE)
        if pretrained_state is not None:
            model.load_encoder(pretrained_state)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        pos_of = {int(o): p for p, o in enumerate(tr_idx)}
        pos_of_all = {int(o): p for p, o in enumerate(tr_idx)}

        def predict_ids(ids, m=model):
            m.eval()
            out = np.empty(len(ids), dtype=np.float32)
            with torch.no_grad():
                for i in range(0, len(ids), 256):
                    bi = ids[i:i + 256]
                    graphs = [row_to_graph[int(b)] for b in bi]
                    batch = to_pyg(graphs).to(DEVICE)
                    twin = torch.tensor(twin_train[bi], dtype=torch.float)
                    p = m(batch, twin=twin).cpu().numpy()
                    for j, b in enumerate(bi):
                        ti = TARGET_IDX[T[b]]
                        mu, sd = stats[T[b]]
                        out[i + j] = p[j, ti] * sd + mu
            return out

        best, best_r2, pat = None, -np.inf, 0
        for ep in range(MAX_EPOCHS):
            model.train()
            perm = np.random.permutation(len(fit_ids))
            for i in range(0, len(perm), BS):
                bi = fit_ids[perm[i:i + BS]]
                idxs = [pos_of_all[int(b)] for b in bi]
                yb = torch.tensor(y_norm[idxs]).unsqueeze(1).to(DEVICE)
                wb = torch.tensor([row_to_graph[int(b)].w.item() for b in bi],
                                  dtype=torch.float).unsqueeze(1).to(DEVICE)
                graphs = [row_to_graph[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_train[bi], dtype=torch.float)
                opt.zero_grad()
                pred = model(batch, twin=twin)
                ti = torch.tensor([TARGET_IDX[T[b]] for b in bi], device=DEVICE)
                pred_sel = pred.gather(1, ti.unsqueeze(1))
                loss = (F.mse_loss(pred_sel, yb, reduction="none") * wb).mean()
                loss.backward(); opt.step()
            hp = predict_ids(ho_ids)
            hr = r2_score(Y[ho_ids], hp)
            if hr > best_r2:
                best_r2 = hr
                best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                pat = 0
            else:
                pat += 1
                if pat >= PATIENCE:
                    break
        model.load_state_dict(best)
        mt_oof_all[va_idx] = predict_ids(va_idx)
        # test prediction via graphs
        model.eval()
        with torch.no_grad():
            te_pred = np.zeros(len(Xte), dtype=np.float32)
            for i in range(0, len(Xte), 256):
                bi = np.arange(i, min(i + 256, len(Xte)))
                graphs = [test_graphs[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_test[bi], dtype=torch.float)
                p = model(batch, twin=twin).cpu().numpy()
                for j, b in enumerate(bi):
                    ttt = tef["target_type"].iloc[int(b)]
                    ti = TARGET_IDX[ttt]
                    mu, sd = stats[ttt]
                    te_pred[i + j] = p[j, ti] * sd + mu
        mt_test_folds[:, f] = te_pred
        print(f"seed {seed}  fold {f}: holdout R2={best_r2:.4f} ({time.time()-t0f:.0f}s)", flush=True)
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    assert not np.isnan(mt_oof_all).any()
    return mt_oof_all, mt_test_folds.mean(axis=1)


print("GNN_SEEDS =", GNN_SEEDS, flush=True)
mt_oof_sum = np.zeros(len(X), dtype=np.float32)
mt_test_sum = np.zeros(len(Xte), dtype=np.float32)
for _gs in GNN_SEEDS:
    _oo, _mt = run_gnn_seed(_gs)
    mt_oof_sum += _oo
    mt_test_sum += _mt
mt_oof_all = mt_oof_sum / len(GNN_SEEDS)
mt_test = mt_test_sum / len(GNN_SEEDS)
assert not np.isnan(mt_oof_all).any()

mt_oof = {t: mt_oof_all[idx_of_target[t]] for t in TARGETS}


# =====================================================================
# Per-target fallback vs the GBM trio stack (Ridge on lgb+xgb+cb).
# =====================================================================
print("\n=== GBM trio stack OOF (fallback floor) ===", flush=True)
gbm_oof = {t: {m: np.zeros(len(idx_of_target[t])) for m in ('lgb', 'xgb', 'cb')}
           for t in TARGETS}
gbm_test = {t: {m: np.zeros(len(Xte)) for m in ('lgb', 'xgb', 'cb')} for t in TARGETS}
import xgboost as xgb
import catboost as cb

for t in TARGETS:
    idx = idx_of_target[t]
    Xt, yt, gt = Xs[idx], Y[idx], G[idx]
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(Xt, yt, gt):
        fit_ids, ho_ids = early_split(tr_idx)
        l = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03,
                              num_leaves=15, min_child_samples=10, subsample=0.8,
                              colsample_bytree=0.8, random_state=SEED, verbose=-1)
        x = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                             random_state=SEED, verbosity=0)
        c = cb.CatBoostRegressor(iterations=2000, learning_rate=0.03, depth=6,
                                 random_seed=SEED, task_type='CPU', verbose=False,
                                 allow_writing_files=False)
        for m, est in ((l, l), (x, x), (c, c)):
            est.fit(Xt[fit_ids], yt[fit_ids], eval_set=[(Xt[ho_ids], yt[ho_ids])])
        gbm_oof[t]['lgb'][va_idx] = l.predict(Xt[va_idx])
        gbm_oof[t]['xgb'][va_idx] = x.predict(Xt[va_idx])
        gbm_oof[t]['cb'][va_idx] = c.predict(Xt[va_idx])
        gbm_test[t]['lgb'] += l.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['xgb'] += x.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['cb'] += c.predict(Xtes) / GLOBAL_FOLDS
    print(f"  {t} done", flush=True)

from sklearn.linear_model import Ridge

stack_oof = {}
stack_test = {}
for t in TARGETS:
    idx = idx_of_target[t]
    yt = Y[idx]; gt = G[idx]
    M = np.column_stack([gbm_oof[t][m] for m in ('lgb', 'xgb', 'cb')])
    Mte = np.column_stack([gbm_test[t][m] for m in ('lgb', 'xgb', 'cb')])
    oof = np.zeros(len(idx)); te_pred = np.zeros(len(Xte))
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(M, yt, gt):
        r = Ridge(alpha=1.0).fit(M[tr_idx], yt[tr_idx])
        oof[va_idx] = r.predict(M[va_idx])
        te_pred += r.predict(Mte) / GLOBAL_FOLDS
    stack_oof[t] = oof; stack_test[t] = te_pred

## 5. Recompute the P14 blend arms in-kernel (no npz, no externals)

In [ ]:
# P14 arms on global indices, recomputed from CORE_B outputs (stack_oof/mt_oof
# per target, stack_test/mt_test per test row). These are the P14 arms the
# v22 gate compares against.
oof_gbm_global = np.full(len(X), np.nan, dtype=np.float32)
oof_mt_global = np.full(len(X), np.nan, dtype=np.float32)
for t in TARGETS:
    idx = idx_of_target[t]
    oof_gbm_global[idx] = stack_oof[t]
    oof_mt_global[idx] = mt_oof[t]
test_gbm_global = np.zeros(len(Xte), dtype=np.float32)
test_mt_global = np.zeros(len(Xte), dtype=np.float32)
for t in TARGETS:
    m_te = (tef["target_type"] == t).values
    test_gbm_global[m_te] = stack_test[t][m_te]
    test_mt_global[m_te] = mt_test[m_te]
assert not np.isnan(oof_gbm_global).any() and not np.isnan(oof_mt_global).any()
print("P14 arms recomputed in-kernel:",
      oof_gbm_global.shape, test_gbm_global.shape, flush=True)


## v22 BERT-arm module sources (inlined verbatim)

In [ ]:
import re
from collections import Counter

import numpy as np

SPECIALS = ("[PAD]", "[CLS]", "[MASK]", "[UNK]")
PROTECTED = ("*", "(", ")")
_BRACKET = re.compile(r"(\[[^\]]+\])")


def segment(smiles, protect=PROTECTED):
    """Initial atomic token units. Protected tokens are always their own unit."""
    out = []
    i = 0
    s = smiles
    while i < len(s):
        if s[i] == "[":
            m = _BRACKET.match(s, i)
            if m:
                out.append(m.group(1))
                i = m.end()
                continue
        two = s[i:i + 2]
        if two in ("Br", "Cl", "Si"):
            out.append(two)
            i += 2
            continue
        ch = s[i]
        if ch in protect:
            out.append(ch)          # never merged with anything
        else:
            out.append(ch)
        i += 1
    return out


def _stratified_subset(smiles, max_subset, seed):
    if len(smiles) <= max_subset:
        return list(smiles)
    rng = np.random.default_rng(seed)
    lens = np.array([len(s) for s in smiles])
    buckets = np.digitize(lens, np.quantile(lens, [0.25, 0.5, 0.75]))
    per = max(1, int(np.ceil(max_subset / len(np.unique(buckets)))))
    chosen = []
    for b in np.unique(buckets):
        idx = np.where(buckets == b)[0]
        take = min(per, idx.size)
        chosen.extend(rng.choice(idx, take, replace=False).tolist())
    # trim if over budget
    rng.shuffle(chosen)
    return [smiles[k] for k in chosen[:max_subset]]


def _apply_merge(tokens, a, b):
    nt = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and tokens[i] == a and tokens[i + 1] == b:
            nt.append(a + b)
            i += 2
        else:
            nt.append(tokens[i])
            i += 1
    return nt


def learn_bpe(smiles, vocab_target=4000, protect=PROTECTED, seed=42,
              max_subset=150000):
    corpus = [_stratified_subset(smiles, max_subset, seed)]
    pieces = [segment(s) for s in corpus[0]]
    merges = []
    vocab = list(SPECIALS)
    seen = set(SPECIALS)
    for toks in pieces:
        for t in toks:
            if t not in seen:
                seen.add(t)
                vocab.append(t)
    while len(merges) < vocab_target:
        pairs = Counter()
        for toks in pieces:
            for a, b in zip(toks, toks[1:]):
                if a in protect or b in protect:
                    continue
                pairs[(a, b)] += 1
        if not pairs:
            break
        (a, b), cnt = pairs.most_common(1)[0]
        if cnt < 2:
            break
        pieces = [_apply_merge(toks, a, b) for toks in pieces]
        merged = a + b
        merges.append((a, b))
        if merged not in seen:
            seen.add(merged)
            vocab.append(merged)
    tok2id = {t: i for i, t in enumerate(vocab)}
    id2tok = {i: t for i, t in enumerate(vocab)}
    return {"tok2id": tok2id, "id2tok": id2tok, "merges": merges,
            "protect": tuple(protect)}


def encode(smiles, tok):
    toks = segment(smiles, tok["protect"])
    for a, b in tok["merges"]:
        toks = _apply_merge(toks, a, b)
    unk = tok["tok2id"]["[UNK]"]
    return [tok["tok2id"].get(t, unk) for t in toks]


def decode(ids, tok):
    special_ids = {tok["tok2id"][s] for s in SPECIALS}
    return "".join(tok["id2tok"][i] for i in ids if i not in special_ids)


def tokenize_batch(tok, smiles, max_len=128):
    cls, pad = tok["tok2id"]["[CLS]"], tok["tok2id"]["[PAD]"]
    out = []
    for sm in smiles:
        row = [cls] + encode(sm, tok)
        if len(row) > max_len:
            row = row[:max_len]
        row = row + [pad] * (max_len - len(row))
        out.append(row)
    return np.asarray(out, dtype=np.int32)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

MASK_ID = 2


class BertEncoder(nn.Module):
    """In-notebook BERT-style MLM encoder (pure torch, no external libs)."""

    def __init__(self, vocab, d=384, layers=6, heads=8, ff=None, max_len=128,
                 dropout=0.1):
        super().__init__()
        self.vocab = vocab
        self.d = d
        self.max_len = max_len
        ff = ff or 4 * d
        self.embed = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d))
        nn.init.normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=heads, dim_feedforward=ff, dropout=dropout,
            activation="gelu", batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=layers)
        self.head = nn.Linear(d, vocab)

    def reset_parameters(self):
        """Deterministically re-initialize all weights from the current RNG seed."""
        self.embed.reset_parameters()
        nn.init.normal_(self.pos, std=0.02)
        for layer in self.enc.layers:
            nn.init.xavier_uniform_(layer.self_attn.in_proj_weight)
            nn.init.constant_(layer.self_attn.in_proj_bias, 0.0)
            nn.init.xavier_uniform_(layer.self_attn.out_proj.weight)
            nn.init.constant_(layer.self_attn.out_proj.bias, 0.0)
            layer.linear1.reset_parameters()
            layer.linear2.reset_parameters()
            layer.norm1.reset_parameters()
            layer.norm2.reset_parameters()
        self.head.reset_parameters()

    def forward(self, ids, mask=None):
        x = self.embed(ids) + self.pos[:, :ids.size(1)]
        h = self.enc(x, src_key_padding_mask=mask)
        logits = self.head(h)
        return logits, h


def pool_embeddings(model, ids, max_len=None):
    """Mean-pool of final-layer hidden states over non-[PAD] tokens -> (n, d)."""
    ids = torch.as_tensor(ids, dtype=torch.long)
    device = next(model.parameters()).device
    ids = ids.to(device)
    if max_len is not None:
        ids = ids[:, :max_len]
    model.eval()
    with torch.no_grad():
        _, h = model(ids)
    h = h.float().cpu().numpy()
    valid = (ids != 0).cpu().numpy().astype(bool)
    mask = valid[..., None]
    sums = (h * mask).sum(axis=1)
    counts = mask.sum(axis=1)[:, 0]
    counts = np.where(counts == 0, 1.0, counts)
    return (sums / counts[:, None]).astype(np.float32)


def _eval_nll(model, ids, mask_p, mask_id, protected_ids, bs):
    model.eval()
    total, cnt = 0.0, 0
    ids_t = torch.as_tensor(ids, dtype=torch.long, device=next(model.parameters()).device)
    for start in range(0, ids_t.size(0), bs):
        batch = ids_t[start:start + bs]
        torch.manual_seed(0)
        rand = torch.rand(batch.cpu().shape)
        maskable = (batch != 0) & (batch != 1) & (batch != mask_id)
        for pid in protected_ids:
            maskable = maskable & (batch != pid)
        to_mask = (rand < mask_p).to(batch.device) & maskable
        if not to_mask.any():
            continue
        targets = torch.full_like(batch, -100)
        targets[to_mask] = batch[to_mask]
        masked = batch.clone()
        masked[to_mask] = mask_id
        logits, _ = model(masked)
        nll = F.cross_entropy(logits.reshape(-1, model.vocab),
                              targets.reshape(-1), reduction="none")
        keep = (targets.reshape(-1) != -100)
        total += float(nll[keep].sum().item())
        cnt += int(keep.sum().item())
    model.train()
    return total / max(1, cnt)


def pretrain_mlm(model, ids, epochs=1, bs=256, lr=3e-4, seed=42, mask_p=0.15,
                 mask_id=MASK_ID, protected_ids=(), val_ids=None, device="cpu"):
    """Masked-token prediction with cosine LR + best-val checkpoint restore.

    Returns (losses, best_val_nll). Protected and special tokens are never
    masked. When val_ids is provided, the model is restored to the state with
    the lowest val NLL (deterministic eval mask). When val_ids is None the
    second return value is the losses list itself.
    """
    rng_state = torch.random.get_rng_state()
    torch.manual_seed(seed)
    if hasattr(model, "reset_parameters"):
        model.reset_parameters()
    model.to(device)
    ids_t = torch.as_tensor(ids, dtype=torch.long, device=device)
    n = ids_t.size(0)
    opt = torch.optim.AdamW(model.parameters(), lr=5.0 * lr)
    steps_per_epoch = max(1, int(np.ceil(n / bs)))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=2 * epochs * steps_per_epoch)
    gen = torch.Generator().manual_seed(seed)

    losses = []
    best_state, best_nll = None, float("inf")
    for _ in range(epochs):
        perm = torch.randperm(n, generator=gen)
        for start in range(0, n, bs):
            batch = ids_t[perm[start:start + bs]]
            rand = torch.rand(batch.shape, generator=gen)
            maskable = (batch != 0) & (batch != 1) & (batch != mask_id)
            for pid in protected_ids:
                maskable = maskable & (batch != pid)
            to_mask = (rand < mask_p) & maskable
            if not to_mask.any():
                losses.append(0.0)
                continue
            targets = torch.full_like(batch, -100)
            targets[to_mask] = batch[to_mask]
            masked = batch.clone()
            masked[to_mask] = mask_id
            logits, _ = model(masked)
            loss = F.cross_entropy(logits.reshape(-1, model.vocab),
                                   targets.reshape(-1), ignore_index=-100)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        if val_ids is not None:
            vn = _eval_nll(model, val_ids, mask_p, mask_id, protected_ids, bs)
            if vn < best_nll:
                best_nll = vn
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    torch.random.set_rng_state(rng_state)
    return losses, (best_nll if best_state is not None else losses)


In [ ]:
"""Fold-safe per-target Ridge "bert" arm on a frozen, label-free pool.

compute_bert_arm() produces out-of-fold (OOF) and test predictions from per-
target Ridge heads fit on frozen encoder embeddings. The pool carries no
labels, so no label leakage flows through the encoder; fold safety comes from
a single shared GroupKFold on canonical SMILES (group = smiles), which never
puts the same polymer on both sides of a split.
"""

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold


def compute_bert_arm(pool_tr, pool_te, y, tt_tr, tt_te, g, n_splits=5, seed=42):
    """Per-target Ridge heads on a frozen pool, folded on canonical SMILES.

    Parameters
    ----------
    pool_tr : (n_tr, d) float32 — frozen, LABEL-FREE embeddings of train rows.
    pool_te : (n_te, d) float32 — frozen embeddings of test rows.
    y : (n_tr,) float32 — train target values, one per row.
    tt_tr : (n_tr,) str array — target-type per train row (eea/egb/egc/ei/eps/nc/tg).
    tt_te : (n_te,) str array — target-type per test row.
    g : (n_tr,) — group ids = canonical smiles per train row (GroupKFold key).
    n_splits : int — number of folds for the ONE shared GroupKFold (default 5).
    seed : int — accepted for API stability / downstream reuse; GroupKFold is
                 deterministic by construction.

    Returns
    -------
    (oof_bert, test_bert) : (n_tr,) float32, (n_te,) float32
        oof_bert[i] is predicted by a head trained only on rows of target
        tt_tr[i] whose fold differs from row i's fold (leak-safe by
        construction). test_bert[i] = mean over fold heads of target tt_te[i].

    Fallbacks (documented):
    * Target with fewer than n_splits rows: one head is fit on ALL of the
      target's rows and used for both OOF and test.
    * Single-row (or empty) target: no Ridge is fit on < 2 samples; the row's
      value is its own label (mean of its rows; per-row target mean).
    * Rows that GroupKFold cannot cover cross-validator-wise (e.g. all rows of
      a target share one group): a head fit on all of that target's rows fills
      the gap. This is unusual and slightly in-sample for those rows only; it
      exists to guarantee no NaN on degenerate inputs.
    * Test rows whose target never appears in train: filled with the global
      train mean (never NaN).
    """
    n_tr = pool_tr.shape[0]
    n_te = pool_te.shape[0]
    oof = np.full(n_tr, np.nan, dtype=np.float64)
    test = np.full(n_te, np.nan, dtype=np.float64)

    global_mean = float(np.mean(y)) if n_tr else 0.0

    # GroupKFold cannot form n_splits folds from fewer groups; on such a
    # degenerate pool (e.g. every row in one smiles group) skip the shared CV
    # and fall back to per-target all-rows heads below. No leak: this only
    # triggers when cross-validation folds are impossible.
    n_groups = len(np.unique(g)) if n_tr else 0
    degenerate = n_groups < n_splits
    if n_tr >= 2 and not degenerate:
        fold = np.empty(n_tr, dtype=np.int64)
        gkf = GroupKFold(n_splits=n_splits)
        for f, (_, va) in enumerate(gkf.split(np.arange(n_tr), y, g)):
            fold[va] = f
    else:
        fold = np.zeros(n_tr, dtype=np.int64)

    for t in np.unique(tt_tr):
        tr_idx = np.where(tt_tr == t)[0]
        n_t = tr_idx.size
        te_idx = np.where(tt_te == t)[0]

        if n_t == 0:
            test[te_idx] = global_mean
            continue

        if n_t < n_splits or degenerate:
            # small-target fallback: one head on all rows, used for both sides
            # (also the whole-pool fallback when groups < n_splits)
            if n_t < 2:
                val = float(np.mean(y[tr_idx]))
                oof[tr_idx] = val
                test[te_idx] = val
            else:
                ridge = Ridge(alpha=1.0, fit_intercept=True)
                ridge.fit(pool_tr[tr_idx], y[tr_idx])
                oof[tr_idx] = ridge.predict(pool_tr[tr_idx])
                if te_idx.size:
                    test[te_idx] = ridge.predict(pool_te[te_idx])
            continue

        # main path: 5-fold-per-row heads (exclusive folds, leak-safe)
        fold_t = fold[tr_idx]
        heads = []
        for f in range(n_splits):
            trf = tr_idx[fold_t != f]
            if trf.size < 2:
                continue
            ridge = Ridge(alpha=1.0, fit_intercept=True)
            ridge.fit(pool_tr[trf], y[trf])
            heads.append((f, ridge))

        if heads:
            for f, ridge in heads:
                va = tr_idx[fold_t == f]
                if va.size:
                    oof[va] = ridge.predict(pool_tr[va])
            if te_idx.size:
                acc = np.stack([ridge.predict(pool_te[te_idx])
                                for _, ridge in heads])
                test[te_idx] = acc.mean(axis=0)

        # anything still uncovered (e.g. all t-rows share one group)
        missing = tr_idx[np.isnan(oof[tr_idx])]
        if missing.size:
            ridge = Ridge(alpha=1.0, fit_intercept=True)
            ridge.fit(pool_tr[tr_idx], y[tr_idx])
            oof[missing] = ridge.predict(pool_tr[missing])
            if not te_idx.size or np.isnan(test[te_idx]).any():
                test[te_idx] = ridge.predict(pool_te[te_idx]) if te_idx.size else test[te_idx]

    # catch-all: no NaN, ever
    test[np.isnan(test)] = global_mean
    oof[np.isnan(oof)] = global_mean

    return (np.asarray(oof, dtype=np.float32),
            np.asarray(test, dtype=np.float32))


def compute_bert_only_r2(oof_bert, y, tt_tr):
    """Gate-0 diagnostic: per-target OOF R^2 of the BERT arm Ridge alone."""
    out = {}
    for t in np.unique(tt_tr):
        idx = np.where(tt_tr == t)[0]
        if idx.size < 2:
            out[t] = 0.0
            continue
        out[t] = float(np.corrcoef(y[idx], oof_bert[idx])[0, 1]) ** 2
    return out


In [ ]:
"""Per-target n-arm Ridge blend (P14 fold_safe_blend generalized to k arms)."""

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold

ALPHA_GRID = [0.1, 0.5, 1.0, 2.5, 5.0, 10.0, 25.0]


def blend_narm_oof(M, y, g, alphas=ALPHA_GRID, n_splits=5):
    """Fold-safe n-arm blend per-target.

    Returns (oof, best_alpha, coefs_mean). coefs_mean has length M.shape[1].
    """
    M = np.asarray(M, dtype=float)
    y = np.asarray(y, dtype=float)
    g = np.asarray(g)
    n = len(y)
    k = M.shape[1] if M.ndim == 2 else 0
    if n < 2:
        return y.copy(), float(alphas[0]), np.zeros(k)
    M = np.where(np.isfinite(M), M, np.nanmean(M, axis=0))
    M = np.where(np.isfinite(M), M, 0.0)
    if len(np.unique(g)) < n_splits:
        lr = Ridge(alpha=alphas[0]).fit(M, y)
        return lr.predict(M), float(alphas[0]), lr.coef_
    cv = list(GroupKFold(n_splits=n_splits).split(M, y, g))
    best, besta = -np.inf, alphas[0]
    for a in alphas:
        o = np.zeros(n)
        for tr, vk in cv:
            o[vk] = Ridge(alpha=a).fit(M[tr], y[tr]).predict(M[vk])
        r = r2_score(y, o)
        if r > best:
            best, besta = r, a
    oof = np.zeros(n)
    coefs = []
    for tr, vk in cv:
        lr = Ridge(alpha=besta).fit(M[tr], y[tr])
        oof[vk] = lr.predict(M[vk])
        coefs.append(lr.coef_)
    return oof, float(besta), np.mean(coefs, axis=0)


def _p14_2arm_oof(M2, y, g, n_splits=5):
    """Fold-safe 2-arm (gbm, mt) OOF alpha scan — P14 reference protocol."""
    M2 = np.asarray(M2, dtype=float)
    y = np.asarray(y, dtype=float)
    g = np.asarray(g)
    n = len(y)
    if n < 2:
        return y.copy()
    M = np.where(np.isfinite(M2), M2, np.nanmean(M2, axis=0))
    M = np.where(np.isfinite(M), M, 0.0)
    if len(np.unique(g)) < n_splits:
        return Ridge(alpha=ALPHA_GRID[0]).fit(M, y).predict(M)
    cv = list(GroupKFold(n_splits=n_splits).split(M, y, g))
    best, out = -np.inf, np.zeros(n)
    for a in ALPHA_GRID:
        o = np.zeros(n)
        for tr, vk in cv:
            o[vk] = Ridge(alpha=a).fit(M[tr], y[tr]).predict(M[vk])
        r = r2_score(y, o)
        if r > best:
            best, out = r, o.copy()
    return out


In [ ]:
"""Pure gate-evaluation + submission-writer for the v22 gate (gates 0-3)."""

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

EPS_NC_EI = ("eps", "nc", "ei")
SOFT_DELTA = 0.0015
STRONG_DELTA = 0.003
WORST_TOL = 0.003

SUBMISSION_ROWS = 4940


def gate_report(p14, v22, leak_count=0, bert_only_r2=None):
    """Evaluate pre-registered gates 0-3.

    p14 / v22 : dict target -> r2 (P14 2-arm reference / v22 3-arm blend).
    Returns dict {gate0, gate1, gate2_soft, gate2_strong, gate3, pass}.
    """
    if not p14:
        return {"gate0": {}, "gate1": int(leak_count),
                "gate2_soft": False, "gate2_strong": False,
                "gate3": False, "pass": False}
    deltas = {t: round(float(v22[t] - p14[t]), 12) for t in p14}
    eps = [t for t in EPS_NC_EI if t in deltas]
    eps_mean = round(float(np.mean([deltas[t] for t in eps])) if eps else 0.0, 12)
    overall = round(float(np.mean(list(deltas.values()))), 12)
    worst = round(float(min(deltas.values())), 12)
    gate2_soft = eps_mean >= SOFT_DELTA and overall >= SOFT_DELTA
    gate2_strong = eps_mean >= STRONG_DELTA and overall >= STRONG_DELTA
    gate3 = worst >= -WORST_TOL
    return {
        "gate0": dict(bert_only_r2) if bert_only_r2 else {},
        "gate1": int(leak_count),
        "gate2_soft": bool(gate2_soft),
        "gate2_strong": bool(gate2_strong),
        "gate3": bool(gate3),
        "pass": bool(leak_count == 0 and gate2_soft and gate3),
    }


def write_submission(df, path):
    """Write the v22 submission in P14 format (id,target; 4940 rows)."""
    if list(df.columns) != ["id", "target"]:
        raise ValueError(
            f"submission frame must have columns ['id','target'], got "
            f"{list(df.columns)}")
    if len(df) != SUBMISSION_ROWS:
        raise ValueError(
            f"submission must have exactly {SUBMISSION_ROWS} rows, got {len(df)}")
    df.to_csv(path, index=False)
    return path


def gate_1_leak_audit(feat_matrix, trf, idx_of_target, folds):
    """v19-style leak audit over the blend input columns.

    Counts rows where any arm feature exactly equals a true other-target
    label of the same polymer (canonical smiles group). Must be 0.
    """
    Y = trf["target"].values
    T = trf["target_type"].values
    G = trf["canon"].values
    n = len(Y)
    F = np.asarray(feat_matrix, dtype=float)
    gkf = GroupKFold(n_splits=folds)
    row_fold = np.zeros(n, dtype=int)
    for f, (_, va) in enumerate(gkf.split(np.zeros((n, 1)), Y, G)):
        row_fold[va] = f
    polymer_rows = {}
    for i in range(n):
        polymer_rows.setdefault(G[i], []).append(i)
    matches = 0
    for f in range(folds):
        for i in np.where(row_fold == f)[0]:
            ti = T[i]
            others = [j for j in polymer_rows[G[i]] if T[j] != ti]
            if not others:
                continue
            labels = set(float(Y[j]) for j in others)
            if any(float(v) in labels for v in F[i]):
                matches += 1
    return int(matches)


## v22 BERT arm + pre-registered gate (submission_v22.csv only on PASS)

In [ ]:
# ---- v22 BERT arm + pre-registered gates 0-3 (port of run_v22_gate.py) ----
# The P14 arms (oof_gbm_global/oof_mt_global/test_gbm_global/test_mt_global)
# were recomputed in-kernel above; G/Y/idx_of_target/trf come from the P14
# fork cells. Gates mirror v21 exactly (EPS_NC_EI, SOFT/STRONG/WORST deltas,
# corr2, gate_1_leak_audit).

def corr2(y, o):
    """Per-target R^2 reported for the gate comparison (corr^2, as the v21 gate)."""
    return float(np.corrcoef(y, o)[0, 1]) ** 2

# ---- BPE tokenizer on a PI1M sample (label-free) ----
p1_path = find_input(INP, "PI1M.csv")
p1_smiles = pd.read_csv(p1_path, nrows=V22_PI_COUNT)
smi_col = "SMILES" if "SMILES" in p1_smiles.columns else "smiles"
p1_smiles = p1_smiles[smi_col].astype(str).tolist()
print("[v22] learning BPE on", len(p1_smiles), "PI1M rows", flush=True)
tok = learn_bpe(p1_smiles, vocab_target=V22_VOCAB,
                max_subset=V22_BPE_SUBSET, seed=V22_SEED)
print("[v22] vocab:", len(tok["tok2id"]), flush=True)

ids_tr = tokenize_batch(tok, trf["smiles"].values, max_len=128)
ids_te = tokenize_batch(tok, tef["smiles"].values, max_len=128)

protected_ids = tuple(tok["tok2id"].get(p, -1) for p in tok["protect"])
protected_ids = tuple(p for p in protected_ids if p >= 0)
model = BertEncoder(vocab=len(tok["tok2id"]), d=V22_D,
                    layers=V22_LAYERS, heads=V22_HEADS)
pretrain_ids = np.concatenate([ids_tr, ids_te]).astype(np.int64)
n_val = max(1, int(0.05 * len(pretrain_ids)))
pretrain_mlm(model, pretrain_ids[:-n_val], epochs=V22_EPOCHS,
             bs=256, lr=3e-4, seed=V22_SEED, mask_p=0.15,
             protected_ids=protected_ids, val_ids=pretrain_ids[-n_val:],
             device="cpu")

pool_tr = pool_embeddings(model, ids_tr.astype(np.int64))
pool_te = pool_embeddings(model, ids_te.astype(np.int64))
print("pooled embeddings:", pool_tr.shape, pool_te.shape, flush=True)

oof_bert, test_bert = compute_bert_arm(
    pool_tr, pool_te, Y, T, tef["target_type"].values,
    G, n_splits=GLOBAL_FOLDS, seed=V22_SEED)
print("bert arm done:", oof_bert.shape, test_bert.shape, flush=True)

# ---- 3-arm blend (P14 fold-safe alpha sweep) ----
oof_v22 = np.zeros(len(Y))
alphas, w_bert, r2_p14, r2_v22 = {}, {}, {}, {}
for t in TARGETS:
    idx = idx_of_target[t]
    gt = G[idx]
    M3 = np.column_stack([oof_gbm_global[idx], oof_mt_global[idx], oof_bert[idx]])
    oof_v22[idx], a, coefs = blend_narm_oof(M3, Y[idx].astype(np.float64), gt,
                                            n_splits=GLOBAL_FOLDS)
    alphas[t] = a
    w_bert[t] = float(coefs[2])
    r2_v22[t] = corr2(Y[idx], oof_v22[idx])

    M2 = np.column_stack([oof_gbm_global[idx], oof_mt_global[idx]])
    b2 = _p14_2arm_oof(M2, Y[idx].astype(np.float64), gt,
                       n_splits=GLOBAL_FOLDS)
    r2_p14[t] = corr2(Y[idx], b2)

mean_p14 = float(np.mean(list(r2_p14.values())))
assert abs(mean_p14 - 0.8641) <= 0.005, (
    f"recomputed P14 {mean_p14:.4f} deviates from reference 0.8641")
mean_v22 = float(np.mean(list(r2_v22.values())))
deltas = {t: r2_v22[t] - r2_p14[t] for t in TARGETS}
eps_delta = float(np.mean([deltas[t] for t in EPS_NC_EI]))
overall = float(np.mean(list(deltas.values())))
worst = float(min(deltas.values()))

leak_count = gate_1_leak_audit(
    np.column_stack([oof_gbm_global, oof_mt_global, oof_bert]),
    trf, idx_of_target, GLOBAL_FOLDS)
bert_only_r2 = compute_bert_only_r2(oof_bert, Y, T)
report = gate_report(r2_p14, r2_v22, leak_count=leak_count,
                     bert_only_r2=bert_only_r2)

print("\n==" * 34)
print("target   r2_p14    r2_v22   delta    w_BERT   bert_only_r2")
for t in TARGETS:
    print(f"{t:6s} {r2_p14[t]:.4f} {r2_v22[t]:.4f} "
          f"{deltas[t]:+.4f}  {w_bert[t]:+.3f}  {bert_only_r2.get(t, 0.0):+.4f}")
print("-" * 34)
print(f"mean_v22 {mean_v22:.4f}  mean_p14 {mean_p14:.4f}  mean_delta {overall:+.4f}")
print(f"eps/nc/ei delta {eps_delta:+.4f}  worst_delta {worst:+.4f}")
print(f"gate1 leak_count {leak_count}  gate2 soft {report['gate2_soft']}  "
      f"gate3 {report['gate3']}")
print(f"GATE: {'PASS -> v22 proceeds' if report['pass'] else 'FAIL -> P14 stays final'}")
print("==" * 34)

rows = [{"target": t, "r2_p14": r2_p14[t], "r2_v22": r2_v22[t],
         "delta": deltas[t], "alpha": alphas[t],
         "w_bert": w_bert[t], "bert_only_r2": bert_only_r2.get(t, float("nan"))}
        for t in TARGETS]
rows.append({"target": "mean", "r2_p14": mean_p14, "r2_v22": mean_v22,
             "delta": overall, "alpha": float("nan"),
             "w_bert": float("nan"), "bert_only_r2": float("nan")})
pd.DataFrame(rows).round(4).to_csv(os.path.join(OUT, "v22_blend_report.csv"),
                                   index=False)
print("wrote", os.path.join(OUT, "v22_blend_report.csv"), flush=True)

final_te = np.zeros(len(tef))
if report["pass"]:
    test_pred = np.zeros(len(tef))
    for t in TARGETS:
        idx = idx_of_target[t]
        idx_te = np.where(tef["target_type"].values == t)[0]
        M_tr = np.column_stack([oof_gbm_global[idx], oof_mt_global[idx],
                                oof_bert[idx]])
        M_te = np.column_stack([test_gbm_global[idx_te], test_mt_global[idx_te],
                                test_bert[idx_te]])
        lr = Ridge(alpha=alphas[t], fit_intercept=True).fit(M_tr, Y[idx])
        test_pred[idx_te] = lr.predict(M_te)
    final_te = test_pred
    assert np.isfinite(final_te).all(), "NaN in test predictions"
    sub = pd.DataFrame({"id": tef["id"].values, "target": final_te})
    write_submission(sub, os.path.join(OUT, "submission_v22.csv"))
    print("GATE=PASS -> wrote submission_v22.csv", flush=True)
else:
    print("GATE=FAIL -> P14 stays final; no v22 submission", flush=True)
print("v22 DONE", flush=True)
